### Reranking Hybrid Search Statergies

Re-ranking is a second-stage filtering process in retrieval systems, especially in RAG pipelines, where we:

1. First use a fast retriever (like BM25, FAISS, hybrid) to fetch top-k documents quickly.

2. Then use a more accurate but slower model (like a cross-encoder or LLM) to re-score and reorder those documents by relevance to the query.

👉 It ensures that the most relevant documents appear at the top, improving the final answer from the LLM.

In [1]:
from pathlib import Path
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from dotenv import load_dotenv
import os

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


/Users/shihab/Downloads/AI/Courses/01Portfolio/LangChainKrisnaik/playground/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
## Load text file
candidate_paths = [
    Path("09. Hybrid Search Strategies/5. langchain_sample.txt"),
    Path("5. langchain_sample.txt"),
]

file_path = next((path for path in candidate_paths if path.exists()), None)
if file_path is None:
    raise FileNotFoundError("Could not find 5. langchain_sample.txt from the current notebook working directory.")

loader = TextLoader(str(file_path))
raw_docs = loader.load()

# Split text into document chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = splitter.split_documents(raw_docs)
docs


[Document(metadata={'source': '5. langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(metadata={'source': '5. langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.'),
 Document(metadata={'source': '5. langchain_sample.txt'}, page_content='Retrieval-Augmented Generation (RAG) is a powerful technique where external knowledge is retrieved and passed into the prompt to ground LLM responses. LangChain makes it easy to implement RAG using vector databases like FAISS, Chroma, and Pinecone.\nBM25 is a tra

In [3]:
## user query
query="How can i use langchain to build an application with memory and tools?"

In [4]:
### FAISS and multilingual Hugging Face embeddings
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
vectorstore = FAISS.from_documents(docs, embedding_model)
retriever = vectorstore.as_retriever(search_kwargs={"k": 8})


In [5]:
## Thesis note
print("Using a multilingual Hugging Face embedding model for retrieval instead of OpenAI embeddings.")
print("This is a better baseline for Bangla-English retrieval experiments in your thesis.")


Using a multilingual Hugging Face embedding model for retrieval instead of OpenAI embeddings.
This is a better baseline for Bangla-English retrieval experiments in your thesis.


In [6]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x1633dda90>, search_kwargs={'k': 8})

In [7]:
retriever


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x1633dda90>, search_kwargs={'k': 8})

In [8]:
## Prompt and use the LLM
llm = init_chat_model("groq:llama-3.1-8b-instant")
llm


ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x30fa7f890>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x30fce0f10>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [9]:
# Prompt Template
prompt = PromptTemplate.from_template("""
You are a careful reranking assistant for a retrieval system.
Rank the following documents from most to least relevant to the user question.

User Question: "{question}"

Documents:
{documents}

Instructions:
- Judge topical relevance only.
- Return document indices in ranked order.
- Use only the indices that exist in the list.

Output format: comma-separated indices like 1,3,2,4
""")


In [10]:
retrieved_docs=retriever.invoke(query)
retrieved_docs

[Document(id='0b168910-78c5-4dea-a6d5-f17cd9235709', metadata={'source': '5. langchain_sample.txt'}, page_content='LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.'),
 Document(id='84b513e1-6c8e-4ed2-95eb-3f81d0c4193c', metadata={'source': '5. langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.'),
 Document(id='97b79108-2922-454c-bdba-7795db0374a3', metadata={'source': '5. langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applicati

In [11]:
chain=prompt| llm | StrOutputParser()
chain

PromptTemplate(input_variables=['documents', 'question'], input_types={}, partial_variables={}, template='\nYou are a careful reranking assistant for a retrieval system.\nRank the following documents from most to least relevant to the user question.\n\nUser Question: "{question}"\n\nDocuments:\n{documents}\n\nInstructions:\n- Judge topical relevance only.\n- Return document indices in ranked order.\n- Use only the indices that exist in the list.\n\nOutput format: comma-separated indices like 1,3,2,4\n')
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x30fa7f890>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x30fce0f10>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_k

In [12]:
doc_lines = [f"{i+1}. {doc.page_content}" for i, doc in enumerate(retrieved_docs)]
formatted_docs = "\n".join(doc_lines)

In [13]:
doc_lines

['1. LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.',
 '2. LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.',
 '3. LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.',
 '4. FAISS is a popular library used for fast approximate nearest neighbor search in high-dimensional spaces. It supports both flat and comp

In [14]:
formatted_docs

'1. LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.\n2. LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.\n3. LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.\n4. FAISS is a popular library used for fast approximate nearest neighbor search in high-dimensional spaces. It supports both flat and compressed ind

In [15]:
response=chain.invoke({"question":query,"documents":formatted_docs})
response

'To rank the documents from most to least relevant to the user question, I will analyze each document\'s content and compare it to the user question.\n\nThe user question is "How can i use langchain to build an application with memory and tools?"\n\nDocument 1 directly mentions LangChain\'s support for tool integration, memory, and context retention, making it highly relevant.\nDocument 3 mentions LangChain\'s components for memory and agents, which is relevant to the user question.\nDocument 2 mentions LangChain\'s integration with third-party services, which is somewhat relevant but less specific than the first two documents.\nDocument 6 mentions Retrieval-Augmented Generation (RAG) and LangChain, which is somewhat relevant but not directly related to the user question.\nDocument 5 and 4 are not relevant to the user question as they discuss dense retrieval, hybrid retrieval, and FAISS, which are not directly related to building an application with memory and tools in LangChain.\n\nBa

In [16]:
# Step 5: Parse and rerank
indices = [int(x.strip()) - 1 for x in response.split(",") if x.strip().isdigit()]
indices

[0, 1, 2, 1, 5]

In [17]:
retrieved_docs

[Document(id='0b168910-78c5-4dea-a6d5-f17cd9235709', metadata={'source': '5. langchain_sample.txt'}, page_content='LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.'),
 Document(id='84b513e1-6c8e-4ed2-95eb-3f81d0c4193c', metadata={'source': '5. langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.'),
 Document(id='97b79108-2922-454c-bdba-7795db0374a3', metadata={'source': '5. langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applicati

In [18]:
reranked_docs = [retrieved_docs[i] for i in indices if 0 <= i < len(retrieved_docs)]
reranked_docs

[Document(id='0b168910-78c5-4dea-a6d5-f17cd9235709', metadata={'source': '5. langchain_sample.txt'}, page_content='LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.'),
 Document(id='84b513e1-6c8e-4ed2-95eb-3f81d0c4193c', metadata={'source': '5. langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.'),
 Document(id='97b79108-2922-454c-bdba-7795db0374a3', metadata={'source': '5. langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applicati

In [19]:
# Step 6: Show results
print("\n📊 Final Reranked Results:")
for i, doc in enumerate(reranked_docs, 1):
    print(f"\nRank {i}:\n{doc.page_content}")


📊 Final Reranked Results:

Rank 1:
LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.
Memory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.

Rank 2:
LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.

Rank 3:
LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.

Rank 4:
LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This 